# Session 11 — Ensuring Data and Model Integrity using Deepchecks

**Goal:** run an automated battery of sanity checks on both data and model —
duplicate rows, label leakage, train/test overlap, weak segments — using
[Deepchecks](https://deepchecks.com/), the kind of checks that are easy to forget to
write by hand but catch real bugs before they reach production.

## How this differs from Evidently (Session 5)

Evidently answers "has production data drifted from training data?" — a *monitoring*
question, asked repeatedly over time. Deepchecks answers "is this dataset/model
internally sound *right now*?" — a *validation* question, typically run once per
training run or once per PR, similar in spirit to a unit test suite for data.

## Prerequisites

```bash
pip install deepchecks
```
Runs entirely locally.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from deepchecks.tabular import Dataset
from deepchecks.tabular.suites import data_integrity, train_test_validation, model_evaluation

print("deepchecks ready")

## Step 1 — Build a dataset with some deliberate problems

To see the checks actually catch something, we inject: duplicate rows, and a
"leaky" feature that's a near-copy of the target (something that would give
suspiciously perfect accuracy if left in).

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "target"]

rng = np.random.default_rng(0)
df_dirty = df.copy()

duplicate_rows = df_dirty.sample(10, random_state=1)
df_dirty = pd.concat([df_dirty, duplicate_rows], ignore_index=True)

df_dirty["target_leak"] = df_dirty["target"] + rng.normal(0, 0.01, size=len(df_dirty))

print(f"Dataset has {len(df_dirty)} rows ({len(duplicate_rows)} injected duplicates)")
print("Columns:", list(df_dirty.columns))

## Step 2 — Wrap it as a Deepchecks `Dataset` and run the integrity suite

`Dataset` needs to know the label column and (optionally) which features are
categorical. `data_integrity()` bundles checks like duplicate detection, mixed data
types, and single-value columns.

In [ ]:
ds = Dataset(df_dirty, label="target", cat_features=[])

integrity_suite = data_integrity()
integrity_result = integrity_suite.run(ds)

for check_result in integrity_result.results:
    name = check_result.check.name()
    passed = check_result.passed_conditions() if check_result.have_conditions() else "N/A"
    print(f"{name:<40} conditions_passed={passed}")

In [ ]:
integrity_result.save_as_html("data_integrity_report.html")
print("Saved data_integrity_report.html")

## Step 3 — Train/test validation: check for leakage and overlap

`train_test_validation()` catches issues that only show up when comparing two splits:
identical rows appearing in both train and test (train/test leakage), or a feature
whose distribution differs suspiciously between the splits.

In [ ]:
clean_df = df_dirty.drop(columns="target_leak")
train_df, test_df = train_test_split(clean_df, test_size=0.2, random_state=42)

train_ds = Dataset(train_df, label="target", cat_features=[])
test_ds = Dataset(test_df, label="target", cat_features=[])

tt_suite = train_test_validation()
tt_result = tt_suite.run(train_ds, test_ds)

for check_result in tt_result.results:
    print(check_result.check.name())

## Step 4 — Model evaluation suite: catch weak segments and overfitting

Beyond a single accuracy number, `model_evaluation()` checks for **weak segments**
(subgroups of the data the model does noticeably worse on) and a train/test
performance gap that would indicate overfitting — both of which a single aggregate
accuracy metric hides.

In [ ]:
X_train, y_train = train_df.drop(columns="target"), train_df["target"]
X_test, y_test = test_df.drop(columns="target"), test_df["target"]

model = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)

model_suite = model_evaluation()
model_result = model_suite.run(train_ds, test_ds, model)

for check_result in model_result.results:
    print(check_result.check.name())

## Step 5 — Turn a check into a CI gate

Every suite result has a pass/fail condition status — this is the piece that plugs
directly into the GitHub Actions pipeline from Session 10: fail the build if data
integrity checks fail, before a bad dataset ever reaches training.

In [ ]:
all_passed = integrity_result.passed(fail_if_warning=False)
print(f"All data integrity conditions passed: {all_passed}")

if not all_passed:
    print("In a CI job, you would: raise SystemExit(1) here to fail the pipeline.")
else:
    print("Dataset is clean enough to proceed to training.")

## What to try next

* Re-run Step 2 on the *original* (non-dirty) iris dataframe and compare which checks
  now pass — seeing a check go from failing to passing is the clearest way to
  understand what each one actually looks for.
* Add the integrity/train-test suites as an early step in the GitHub Actions workflow
  from Session 10, before the `pytest` step, so bad data fails fast.
* Combine with Evidently (Session 5): Deepchecks for "is this dataset sound", Evidently
  for "has production drifted from what Deepchecks already approved."